<a href="https://colab.research.google.com/github/Rayudu-Somisetty/deep_learning_lab_tasks/blob/main/cancer_data_set_activation_func.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Activation func for cancer dataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer  # Reliable source
import pandas as pd

# Load reliable breast cancer dataset (569 samples, 30 features)
data = load_breast_cancer(as_frame=True).frame  # No 'id', no '?', target=0/1 [web:6][web:9]
print("Dataset shape:", data.shape)
print("Zeros count:", (data.drop('target', axis=1) == 0).sum().sum())  # ~290 zeros OK

X = data.drop(columns=['target']).values
y = data['target'].values

# Better split: 75/25 for more training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y  # Balanced classes
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Threshold activation (step function)
def threshold(x):
    return 1 if x >= 0 else 0

class Perceptron:
    def __init__(self, lr=0.001, epochs=500):  # Tuned: smaller lr, more epochs [web:8]
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = None

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        indices = np.random.permutation(len(X))  # Shuffle to avoid order bias
        for epoch in range(self.epochs):
            for i in indices:  # Stochastic with shuffle
                z = np.dot(X[i], self.w) + self.b
                y_pred = threshold(z)
                update = self.lr * (y[i] - y_pred)
                self.w += update * X[i]
                self.b += update

    def predict(self, X):
        return np.array([threshold(np.dot(x, self.w) + self.b) for x in X])

# Train and evaluate
p = Perceptron(lr=0.001, epochs=500)
p.fit(X_train, y_train)

train_acc = accuracy_score(y_train, p.predict(X_train))
test_acc = accuracy_score(y_test, p.predict(X_test))
print("Perceptron Train Accuracy:", train_acc)
print("Perceptron Test Accuracy:", test_acc)


Dataset shape: (569, 31)
Zeros count: 78
Perceptron Train Accuracy: 1.0
Perceptron Test Accuracy: 0.9300699300699301
